            # eReefs demo notebook

            This notebook demonstrates the refactored `ereefs` toolkit on small synthetic EMS-style NetCDF files. It covers the main documented local workflows:

            - regular and curvilinear grid discovery
            - standard and smoother display maps
            - surface, bottom, and depth-specific time series
            - depth-averaged and depth-below-free-surface extraction
            - vertical profiles
            - transect slices where the requested line does not exactly pass through cell centres
            


In [ ]:
            repo_root <- if (file.exists("DESCRIPTION")) getwd() else normalizePath("..", winslash = "/", mustWork = TRUE)
            .libPaths(c(file.path(repo_root, ".r_libs"), .libPaths()))

            suppressPackageStartupMessages({
              library(pkgload)
              library(dplyr)
              library(ggplot2)
              library(tibble)
            })

            pkgload::load_all(repo_root)
            source(file.path(repo_root, "notebooks", "create_demo_datasets.R"))
            demo_paths <- create_demo_datasets(file.path(repo_root, "notebooks", "demo_data"))
            demo_paths
            


In [ ]:
            notebook_output_dir <- file.path(repo_root, "notebooks", "output", "local_demo")
            dir.create(notebook_output_dir, recursive = TRUE, showWarnings = FALSE)

            save_plot_display <- function(plot_obj, filename, width = 8, height = 6, dpi = 120) {
              out_path <- file.path(notebook_output_dir, filename)
              ggplot2::ggsave(out_path, plot = plot_obj, width = width, height = height, dpi = dpi)
              print(plot_obj)
              out_path
            }
            


            This block inspects the horizontal grid metadata for both demo files so we can confirm that the package is distinguishing regular and curvilinear geometries correctly.
            


In [ ]:
            regular_grids <- get_ereefs_grids(demo_paths$regular)
            curvilinear_grids <- get_ereefs_grids(demo_paths$curvilinear)

            list(
              regular = list(
                grid_type = regular_grids$grid_type,
                x_grid_dim = dim(regular_grids$x_grid),
                y_grid_dim = dim(regular_grids$y_grid),
                n_cells = nrow(regular_grids$spatial_grid)
              ),
              curvilinear = list(
                grid_type = curvilinear_grids$grid_type,
                x_grid_dim = dim(curvilinear_grids$x_grid),
                y_grid_dim = dim(curvilinear_grids$y_grid),
                n_cells = nrow(curvilinear_grids$spatial_grid)
              )
            )
            


            These examples draw the same regular-grid field twice: first as polygons and then with the optional smoother rasterised display mode.
            


In [ ]:
            box <- c(146.2, 147.3, -20.4, -19.2)

            regular_map_polygon <- map_ereefs(
              var_name = "temp",
              target_date = as.Date("2020-01-02"),
              layer = "surface",
              input_file = demo_paths$regular,
              box_bounds = box,
              scale_col = "viridis",
              suppress_print = TRUE,
              return_poly = TRUE,
              label_towns = FALSE
            )

            regular_map_plot <- plot_map(
              regular_map_polygon,
              box_bounds = box,
              scale_col = "viridis",
              suppress_print = TRUE,
              label_towns = FALSE
            )
            save_plot_display(regular_map_plot, "regular_map_polygon.png")

            regular_map_smooth <- plot_map(
              regular_map_polygon,
              box_bounds = box,
              scale_col = "viridis",
              plot_style = "smooth",
              smooth_pixels = 400,
              suppress_print = TRUE,
              label_towns = FALSE,
              var_longname = "Same data, smoother display"
            )
            save_plot_display(regular_map_smooth, "regular_map_smooth.png")
            


            Here we extract time series from the surface, the deepest wet layer, and a fixed depth below mean sea level at one location.
            


In [ ]:
            ts_surface <- get_ereefs_ts(
              var_names = c("temp", "salt"),
              geocoordinates = tibble(latitude = -19.75, longitude = 146.75),
              layer = "surface",
              start_date = as.POSIXct("2020-01-01 00:00:00", tz = "Etc/GMT-10"),
              end_date = as.POSIXct("2020-01-03 00:00:00", tz = "Etc/GMT-10"),
              input_file = demo_paths$regular,
              verbosity = 0
            )

            ts_bottom <- get_ereefs_ts(
              var_names = "temp",
              geocoordinates = tibble(latitude = -19.75, longitude = 146.75),
              layer = "bottom",
              start_date = as.POSIXct("2020-01-01 00:00:00", tz = "Etc/GMT-10"),
              end_date = as.POSIXct("2020-01-03 00:00:00", tz = "Etc/GMT-10"),
              input_file = demo_paths$regular,
              verbosity = 0
            )

            ts_msl <- get_ereefs_ts(
              var_names = "temp",
              geocoordinates = tibble(latitude = -19.75, longitude = 146.75),
              layer = -5,
              start_date = as.POSIXct("2020-01-01 00:00:00", tz = "Etc/GMT-10"),
              end_date = as.POSIXct("2020-01-03 00:00:00", tz = "Etc/GMT-10"),
              input_file = demo_paths$regular,
              verbosity = 0
            )

            list(
              surface_head = head(ts_surface),
              bottom_head = head(ts_bottom),
              depth_below_msl_head = head(ts_msl)
            )
            


            This block demonstrates depth-averaged extraction and the fallback behaviour when a simple-format file does not provide `eta`.
            


In [ ]:
            depth_avg <- get_ereefs_depth_integrated_ts(
              var_names = "temp",
              geocoordinates = tibble(latitude = -19.75, longitude = 146.75),
              start_date = as.POSIXct("2020-01-01 00:00:00", tz = "Etc/GMT-10"),
              end_date = as.POSIXct("2020-01-03 00:00:00", tz = "Etc/GMT-10"),
              input_file = demo_paths$regular,
              verbosity = 0
            )

            eta_warning <- NULL
            depth_free_surface <- withCallingHandlers(
              get_ereefs_depth_specified_ts(
                var_names = "temp",
                geocoordinates = tibble(latitude = -19.75, longitude = 146.75),
                depth = 2,
                start_date = as.POSIXct("2020-01-01 00:00:00", tz = "Etc/GMT-10"),
                end_date = as.POSIXct("2020-01-03 00:00:00", tz = "Etc/GMT-10"),
                input_file = demo_paths$regular_noeta,
                verbosity = 0
              ),
              warning = function(w) {
                eta_warning <<- conditionMessage(w)
                invokeRestart("muffleWarning")
              }
            )

            list(
              depth_average_head = head(depth_avg),
              depth_below_free_surface_head = head(depth_free_surface),
              no_eta_warning = eta_warning
            )
            


            The next example extracts a vertical profile from the curvilinear demo file and plots it at the requested time.
            


In [ ]:
            profile <- get_ereefs_profile(
              var_names = "temp",
              geolocation = c(-19.4, 147.75),
              start_date = as.POSIXct("2020-01-01 00:00:00", tz = "Etc/GMT-10"),
              end_date = as.POSIXct("2020-01-02 00:00:00", tz = "Etc/GMT-10"),
              input_file = demo_paths$curvilinear
            )

            profile_plot <- plot_ereefs_profile(profile, var_name = "temp", target_date = as.Date("2020-01-01"))
            save_plot_display(profile_plot, "curvilinear_profile.png")
            


            Finally, this demo extracts a vertical slice along a transect that does not align exactly with cell centres and plots the result.
            


In [ ]:
            slice <- get_ereefs_slice(
              var_names = "temp",
              geolocation = data.frame(
                latitude = c(-20.05, -18.7),
                longitude = c(147.05, 148.35)
              ),
              target_date = as.POSIXct("2020-01-01 00:00:00", tz = "Etc/GMT-10"),
              input_file = demo_paths$curvilinear
            )

            slice_plot <- plot_ereefs_slice(slice, var_name = "temp", scale_col = "viridis")
            save_plot_display(slice_plot, "curvilinear_slice.png", width = 9, height = 5)
            
